In [1]:
import numpy as np

topsis的目的：寻找各个方案中最优者  
- 方法：寻找与理想最优解距离最近且与理想最劣解最远的方案  
- 理想最优解为各个指标的最优值的汇总  
<img src="img/topsis-1.png" style="zoom:60%;">  
-  数据处理    
1. 标准统一  
在遇到的问题中，可能遇到不同指标的最优值评价标准不同，例如指标A要求越大越好，指标B越小越好，指标C越接近其中一个值最好  
对于这些不同的指标，需要先进行数据处理  
<img src="img/topsis-2.png" style="zoom:70%;">
2. 数据归一  
$\hat{x}_{ij} = \frac{x_{ij}}{\sqrt{\sum_{i=1}^n}x_ij^2}$

In [ ]:
# 手动输入数据并预处理
import numpy as np

print("被评价对象的数目：")
n = eval(input())
print("被评价指标的数目：")
m = eval(input())

print("输入评价类型：1：极大型，2：极小型，3：中间型，4：区间型，你在输入数据时各个数字之间用空格隔开")
kind = [int(x) for x in input().split()]

print("输入数据矩阵：,每行数字之间用空格隔开，输入完一行后回车输入下一行")
A = np.zeros((n, m))
for i in range(n):
    A[i] = input().split()
    A[i] = list(map(float, A[i]))
print("你输入的矩阵为：\n{}" .format(A))

In [ ]:
A = np.array([[90,4.0,35,15],[85,3.0,50,21],[92,4.5,40,19],[88,3.5,38,17]])
kind = [1,2,3,4]

#### 不同类型的数据的处理方法

In [ ]:
# 极小型
def min_to_max(x):
    maxx = np.max(x)
    hat_x = maxx - x  
    return hat_x

print(min_to_max(np.array([1, 2, 3, 4, 5])))

[4 3 2 1 0]


In [ ]:
# 中间型
def best_to_max(x):
    best = float(input("输入最优值："))
    M = np.max(abs(x - best))
    if M == 0:
        return np.ones_like(x, dtype=float)
    hat_x = 1 - abs(x - best) / M
    return hat_x

print(best_to_max(np.array([1, 2, 3, 4, 5])))


[0.  0.5 1.  0.5 0. ]


In [ ]:
# 区间型
def fuzzy_interval_membership(x):
    lower, upper = map(float, input("输入区间下限和上限，用空格隔开：").split())
    left_dist = lower - x        
    right_dist = x - upper       
    dist = np.maximum(left_dist, 0) + np.maximum(right_dist, 0)
    M = np.max(dist)
    if M == 0:
        return np.ones_like(x, dtype=float)
    hat_x = 1 - dist / M
    
    return hat_x

print(fuzzy_interval_membership(np.array([1, 2, 3, 4, 5])))

[0.5 1.  1.  0.5 0. ]


In [ ]:
# 数据正向化，归一化
def positive_orientation(A, kinds):
    n, m = A.shape
    X = np.zeros((n, m))
    
    for i in range(m):
        col = A[:, i]
        kind = kinds[i]
        
        if kind == 1:
            X[:, i] = col
        elif kind == 2:
            X[:, i] = min_to_max(col)
        elif kind == 3:
            X[:, i] = best_to_max(col)
        elif kind == 4:
            X[:, i] = fuzzy_interval_membership(col)
        else:
            raise ValueError(f"第 {i+1} 个指标的类型 {kind} 无效，请使用 1-4")
    return X

def normalize(X):
    col_sumsq = np.sum(X**2, axis=0)
    col_norms = np.sqrt(col_sumsq)
    col_norms[col_norms == 0] = 1
    
    return X / col_norms

X = normalize(positive_orientation(A, kind))

In [ ]:
# 接受格列的权重，这里手动输入
print("输入权重向量，每个数字之间用空格隔开")
w = [float(x) for x in input().split()]
print("你输入的权重向量为：\n{}" .format(w))
w = np.array(w)

In [2]:
def topsis(X, weights=None):
    n, m = X.shape
    
    if weights is None:
        weights = np.ones(m) / m
    else:
        weights = np.array(weights) / np.sum(weights)  
    weighted_X = X * weights
    
    z_plus = np.max(weighted_X, axis=0)   
    z_minus = np.min(weighted_X, axis=0)  
    
    d_plus = np.sqrt(np.sum((weighted_X - z_plus)**2, axis=1))
    d_minus = np.sqrt(np.sum((weighted_X - z_minus)**2, axis=1))
    scores = d_minus / (d_plus + d_minus)
    
    ranking = np.argsort(-scores)  
    
    return scores, ranking

## 完整代码

In [ ]:
# 数据
A = np.array([[90,4.0,35,15],[85,3.0,50,21],[92,4.5,40,19],[88,3.5,38,17]])
kind = [1,2,3,4]

#### 无熵权法

In [ ]:
import numpy as np

# ==================== 数据 ====================
A = np.array([[90, 4.0, 35, 15],
              [85, 3.0, 50, 21],
              [92, 4.5, 40, 19],
              [88, 3.5, 38, 17]])
kind = [1, 2, 3, 4]

# ==================== 正向化函数（改为参数输入） ====================
def min_to_max(x):
    """极小型指标 -> 极大型指标"""
    maxx = np.max(x)
    hat_x = maxx - x
    return hat_x

def best_to_max(x, best):
    """中间型指标 -> 极大型指标"""
    M = np.max(np.abs(x - best))
    if M == 0:
        return np.ones_like(x, dtype=float)
    hat_x = 1 - np.abs(x - best) / M
    return hat_x

def fuzzy_interval_membership(x, lower, upper):
    """区间型指标 -> 极大型指标"""
    left_dist = np.maximum(lower - x, 0)
    right_dist = np.maximum(x - upper, 0)
    dist = left_dist + right_dist
    M = np.max(dist)
    if M == 0:
        return np.ones_like(x, dtype=float)
    hat_x = 1 - dist / M
    return hat_x

# ==================== 数据正向化（改为参数输入） ====================
def positive_orientation(A, kinds, best_params=None, interval_params=None):
    """
    参数：
        A: 原始数据矩阵，行=方案，列=指标
        kinds: 指标类型列表，1=极大型，2=极小型，3=中间型，4=区间型
        best_params: 字典，键=列索引，值=最优值，如 {2: 40}
        interval_params: 字典，键=列索引，值=(下限, 上限)，如 {3: (15, 20)}
    """
    if best_params is None:
        best_params = {}
    if interval_params is None:
        interval_params = {}
    
    n, m = A.shape
    X = np.zeros((n, m))
    
    for i in range(m):
        col = A[:, i].astype(float)
        kind_val = kinds[i]
        
        if kind_val == 1:
            X[:, i] = col
        elif kind_val == 2:
            X[:, i] = min_to_max(col)
        elif kind_val == 3:
            if i not in best_params:
                raise ValueError(f"第 {i+1} 个指标是中间型，请在 best_params 中提供最优值")
            X[:, i] = best_to_max(col, best_params[i])
        elif kind_val == 4:
            if i not in interval_params:
                raise ValueError(f"第 {i+1} 个指标是区间型，请在 interval_params 中提供区间")
            lower, upper = interval_params[i]
            X[:, i] = fuzzy_interval_membership(col, lower, upper)
        else:
            raise ValueError(f"第 {i+1} 个指标的类型 {kind_val} 无效，请使用 1-4")
    
    return X

# ==================== 归一化 ====================
def normalize(X):
    col_sumsq = np.sum(X**2, axis=0)
    col_norms = np.sqrt(col_sumsq)
    col_norms[col_norms == 0] = 1
    return X / col_norms


# ==================== TOPSIS ====================
def topsis(X, weights=None):
    n, m = X.shape
    if weights is None:
        weights = np.ones(m) / m
    else:
        weights = np.array(weights) / np.sum(weights)

    weighted_X = X * weights
    z_plus = np.max(weighted_X, axis=0)
    z_minus = np.min(weighted_X, axis=0)
    d_plus = np.sqrt(np.sum((weighted_X - z_plus)**2, axis=1))
    d_minus = np.sqrt(np.sum((weighted_X - z_minus)**2, axis=1))
    scores = d_minus / (d_plus + d_minus)
    ranking = np.argsort(-scores) + 1  # 排名从1开始

    return scores, ranking

# ==================== 主流程 ====================
# 设定参数（你只需在这里修改）
best_params = {2: 40}           # 第3个指标(索引2)是中间型，最优值为40
interval_params = {3: (16, 18)}  # 第4个指标(索引3)是区间型，最佳区间[16,18]

# 正向化
X_positive = positive_orientation(A, kind, best_params, interval_params)
print("正向化后的矩阵：\n", X_positive)

# 归一化
X_norm = normalize(X_positive)

# TOPSIS 分析
scores, ranking = topsis(X_norm)
print("\nTOPSIS方法分析结果：")
n = len(scores)
for i in range(n):
    print(f"方案 {i+1} 的得分为：{scores[i]:.4f}，排名为：{ranking[i]}")

正向化后的矩阵：
 [[90.          0.5         0.5         0.66666667]
 [85.          1.5         0.          0.        ]
 [92.          0.          1.          0.66666667]
 [88.          1.          0.8         1.        ]]

TOPSIS方法分析结果：
方案 1 的得分为：0.4899，排名为：4
方案 2 的得分为：0.4378，排名为：3
方案 3 的得分为：0.5110，排名为：1
方案 4 的得分为：0.7788，排名为：2
